In [3]:
from collections import Counter
from abc import ABC, abstractmethod
from typing import Dict, Any, Sequence, Tuple, Optional

import numpy as np

In [4]:
CacheKey = Tuple[int, int, int]  # (vid, layer, tile)

class CachePolicy(ABC):
    @abstractmethod
    def get(self, key: CacheKey) -> Any: ...
    @abstractmethod
    def put(self, key: CacheKey, value: Any, size: int) -> Tuple[bool, list]: ...
    @abstractmethod
    def contains(self, key: CacheKey) -> bool: ...
    @abstractmethod
    def remove(self, key: CacheKey) -> bool: ...
    @abstractmethod
    def clear(self) -> None: ...
    @abstractmethod
    def keys(self): ...
    @abstractmethod
    def stats(self) -> Dict[str, Any]: ...

In [ ]:
class LruPolicy(CachePolicy):
    def __init__(self, max_size: int):
        self.cur_size = 0
        self.max_size = max_size
        self.cache = {}             # (vid, layer, tile) -> (value, size)
        self.access_order = []      # keys in order of access (oldest first)
    
    def get(self, key):
        if key not in self.cache:
            return None

        self.access_order.remove(key)
        self.access_order.append(key)
        
        return self.cache[key][0]
    
    def put(self, key, value, size: int):
        evicted = []

        if key in self.cache:
            old_size = self.cache[key][1]
            self.cur_size -= old_size
            self.access_order.remove(key)
            del self.cache[key]

        while self.cur_size + size > self.max_size and self.access_order:
            lru_key = self.access_order.pop(0)  # Remove oldest (least recently used)
            lru_size = self.cache[lru_key][1]
            self.cur_size -= lru_size
            del self.cache[lru_key]
            evicted.append(lru_key)

        # Add new item if there's space
        if self.cur_size + size <= self.max_size:
            self.cache[key] = (value, size)
            self.access_order.append(key)
            self.cur_size += size

        return evicted
    
    def contains(self, key) -> bool:
        return key in self.cache
    
    def remove(self, key):
        if key not in self.cache:
            return False
        
        size = self.cache[key][1]
        self.cur_size -= size
        self.access_order.remove(key)
        del self.cache[key]

        return True
    
    def clear(self):
        self.cache.clear()
        self.access_order.clear()
        self.cur_size = 0
    
    def get_stats(self) -> Dict[str, Any]:
        return {
            'size': self.cur_size,
            'max_size': self.max_size,
            'num_items': len(self.cache),
            'utilization': self.cur_size / self.max_size if self.max_size > 0 else 0
        }
    
    def keys(self):
        return self.cache.keys()
    
    def stats(self) -> Dict[str, Any]:
        return {
            'size': self.cur_size,
            'max_size': self.max_size,
            'num_items': len(self.cache),
            'utilization': self.cur_size / self.max_size if self.max_size > 0 else 0
        }

In [ ]:
class SvcLruPolicy(LruPolicy):
    def __init__(self, max_size: int):
        super().__init__(max_size)

    def put(self, key, value, size: int):
        evicted = []
        if key in self.cache:
            old_size = self.cache[key][1]
            self.cur_size -= old_size
            self.access_order.remove(key)
            del self.cache[key]
        
        while self.cur_size + size > self.max_size and self.access_order:
            victim_key = None

            for k in self.access_order:
                if len(k) > 1 and k[1] == 1:
                    victim_key = k
                    break

            if victim_key is None:
                victim_key = self.access_order[0]
            
            victim_size = self.cache[victim_key][1]
            self.cur_size -= victim_size
            self.access_order.remove(victim_key)
            del self.cache[victim_key]
            evicted.append(victim_key)            

        if self.cur_size + size <= self.max_size:
            self.cache[key] = (value, size)
            self.access_order.append(key)
            self.cur_size += size
        
        return evicted

In [ ]:
class CacheEngineEnv:
    def __init__(
        self,
        n_users: int = 1000,
        n_tiles: int = 16,
        n_layers: int = 2,
        n_gops: int = 60,
        n_videos: int = 100,
        cache_capacity: float = 100e6,  # capacity in bytes
        policy: CachePolicy | None = None
    ):
        self.max_capacity = cache_capacity
        self.tile_size_bytes = {
            0: 2e6 / n_tiles,   # base layer tile size in bytes
            1: 15e6 / n_tiles   # enhancement layer tile size in bytes
        }
        self.n_users = n_users
        self.n_layers = n_layers
        self.n_videos = n_videos
        self.n_gops = n_gops
        self.n_tiles = n_tiles
        
        # Initialize LRU policy
        self.policy = policy or LruPolicy(max_size=int(cache_capacity))

        self.cache_bitmap = np.zeros(
            (self.n_videos, self.n_layers, self.n_tiles, self.n_gops), dtype=np.int8
        )
        
        self.content_popularity = np.zeros((n_videos,), dtype=np.float32)
        self.user_visited = np.zeros((n_users, n_videos), dtype=bool)

    def _cache_tile(self, vid, layer, tile_idx, gop_idx):
        key = (vid, layer, tile_idx, gop_idx)
        tile_size = self.tile_size_bytes[layer]

        # Use LRU policy - automatically handles eviction
        evicted = self.policy.put(key, None, int(tile_size))

        # Update cache bitmap
        self.cache_bitmap[vid, layer, tile_idx, gop_idx] = 1
        for e_vid, e_layer, e_tile, e_gop in evicted:
            self.cache_bitmap[e_vid, e_layer, e_tile, e_gop] = 0

        return evicted

    def update_content_popularity(self, requests: Sequence[Dict[str, Any]]):
        for req in requests:
            user = req['u']
            vid = req['video']

            if self.user_visited[user, vid]:
                continue

            self.content_popularity[vid] += 1
            self.user_visited[user, vid] = True

    def get_content_popularity(self):
        return self.content_popularity

    def get_video_popularity_norm(self, vid_id: int) -> float:
        total_requests = np.sum(self.content_popularity)
        
        # Avoid division by zero at the start of the episode
        if total_requests == 0:
            return 0.0
            
        # Return probability: P(v) = count(v) / total_count
        return self.content_popularity[vid_id] / total_requests

    def get_tile(self, vid, layer, tile_idx, gop_idx):
        key = (vid, layer, tile_idx, gop_idx)
        if self.policy.contains(key):
            self.policy.get(key)  # Update access order
            return True
        return False
    
    def get_current_capacity(self):
        return self.policy.cur_size
    
    def get_cache_bitmap(self):
        return self.cache_bitmap
    
    def clear_cache(self):
        self.policy.clear()

    def process_cache_prefetching(self, cache_actions):
        for act in cache_actions:
            gop = act['gop']
            vid = act['video']
            tiles = act['tiles']

            for tile_idx, tile in enumerate(tiles):
                self._cache_tile(vid, 0, tile_idx, gop)

            for tile_idx, tile in enumerate(tiles):
                if (
                    not self.policy.contains((vid, 1, tile_idx, gop)) 
                    and tile == 1
                ):
                    self._cache_tile(vid, 1, tile_idx, gop)

        return self.get_cache_bitmap()
    
    def cache_prefetching_(self, cache_actions):
        for act in cache_actions:
            gop = act['gop']
            vid = act['video']
            tiles = act['tiles']

            for tile_idx, tile in enumerate(tiles):
                self._cache_tile(vid, 0, tile_idx, gop)

            for tile_idx, tile in enumerate(tiles):
                if (
                    not self.policy.contains((vid, 1, tile_idx, gop)) 
                    and tile == 1
                ):
                    self._cache_tile(vid, 1, tile_idx, gop)

        return self.get_cache_bitmap()

    def reset(self, **kwargs):
        self.clear_cache()

        self.content_popularity = np.zeros((self.n_videos,), dtype=np.float32)
        self.user_visited = np.zeros((self.n_users, self.n_videos), dtype=bool)
        
        self.cache_bitmap = np.zeros(
            (self.n_videos, self.n_layers, self.n_tiles, self.n_gops), dtype=np.int8
        )

        ### Randomly pre-fill cache ###
        rng = np.random.default_rng()
        keys = [(v, l, t) for v in range(self.n_videos)
                         for l in range(self.n_layers)
                         for t in range(self.n_tiles)]
        rng.shuffle(keys)

        for vid, layer, tile_idx in keys:
            tile_size = int(self.tile_size_bytes[layer])
            if self.policy.cur_size + tile_size > self.max_capacity:
                break
            # Random GOP index for each tile
            gop_idx = rng.integers(0, self.n_gops)
            if not self.policy.contains((vid, layer, tile_idx, gop_idx)):
                self._cache_tile(vid, layer, tile_idx, gop_idx)
        ### End Randomly pre-fill cache ###
        
        
        return None, {"cache": self.get_cache_bitmap()}


In [7]:
# Test LRU Policy with tuple keys (video, layer, tile)
if __name__ == "__main__":
    print("=" * 5, "LRU Cache Policy Test", "=" * 5)
    
    # Create LRU cache with 10 MB capacity
    lru = LruPolicy(max_size=10 * 1024 * 1024)  # 10 MB
    
    print(f"Initial state: {lru.get_stats()}\n")
    
    # Add some items using tuple keys (video, layer, tile)
    print("Adding items:")
    evicted = lru.put(
        (0, 0, 0), 
        "data_v0_l0_t0", 
        2 * 1024 * 1024
    )  # 2 MB
    print(f"  Added (0,0,0) (2 MB), evicted: {evicted}")
    
    evicted = lru.put(
        (1, 0, 0), 
        "data_v1_l0_t0", 
        3 * 1024 * 1024
    )  # 3 MB
    print(f"  Added (1,0,0) (3 MB), evicted: {evicted}")
    
    evicted = lru.put(
        (2, 0, 0), 
        "data_v2_l0_t0", 
        4 * 1024 * 1024
    )  # 4 MB
    print(f"  Added (2,0,0) (4 MB), evicted: {evicted}")
    
    print(f"\nCurrent state: {lru.get_stats()}")
    print(f"Access order (oldest→newest): {lru.access_order}\n")
    
    # Access tile (0,0,0) (moves it to most recent)
    print("Accessing (0,0,0)...")
    data = lru.get((0, 0, 0))
    print(f"  Retrieved: {data}")
    print(f"  New access order: {lru.access_order}\n")
    
    # Add item that requires eviction
    print("Adding large item (3 MB) - should evict LRU item...")
    evicted = lru.put((3, 0, 0), "data_v3_l0_t0", 3 * 1024 * 1024)
    print(f"  Evicted: {evicted}")
    print(f"  Current access order: {lru.access_order}")
    print(f"  State: {lru.get_stats()}\n")
    
    # Test contains
    print("Testing contains:")
    for key in [(0, 0, 0), (1, 0, 0), (2, 0, 0), (3, 0, 0)]:
        print(f"  {key}: {lru.contains(key)}")
    
    print("\n" + "=" * 60)

===== LRU Cache Policy Test =====
Initial state: {'size': 0, 'max_size': 10485760, 'num_items': 0, 'utilization': 0.0}

Adding items:
  Added (0,0,0) (2 MB), evicted: []
  Added (1,0,0) (3 MB), evicted: []
  Added (2,0,0) (4 MB), evicted: []

Current state: {'size': 9437184, 'max_size': 10485760, 'num_items': 3, 'utilization': 0.9}
Access order (oldest→newest): [(0, 0, 0), (1, 0, 0), (2, 0, 0)]

Accessing (0,0,0)...
  Retrieved: data_v0_l0_t0
  New access order: [(1, 0, 0), (2, 0, 0), (0, 0, 0)]

Adding large item (3 MB) - should evict LRU item...
  Evicted: [(1, 0, 0)]
  Current access order: [(2, 0, 0), (0, 0, 0), (3, 0, 0)]
  State: {'size': 9437184, 'max_size': 10485760, 'num_items': 3, 'utilization': 0.9}

Testing contains:
  (0, 0, 0): True
  (1, 0, 0): False
  (2, 0, 0): True
  (3, 0, 0): True



In [11]:
if __name__ == "__main__":
    # Test configuration
    n_tiles = 4
    n_layers = 2
    n_videos = 10
    cache_capacity = 10e6  # 10 MB
    
    print("=" * 5, "Cache Storage Test with LRU Policy (Computed Matrix)", "=" * 5)
    
    # Initialize cache environment with LRU
    cache = CacheEngineEnv(
        n_tiles=n_tiles * n_tiles,
        n_layers=n_layers,
        n_videos=n_videos,
        cache_capacity=cache_capacity
    )
    
    print(f"Configuration:")
    print(f"  Grid size: {n_tiles}x{n_tiles} = {n_tiles*n_tiles} tiles")
    print(f"  Layers: {n_layers}")
    print(f"  Videos: {n_videos}")
    print(f"  Capacity: {cache_capacity/1e6:.1f} MB")
    print(f"  Base layer tile size: {cache.tile_size_bytes[0]/1e6:.3f} MB")
    print(f"  Enhancement tile size: {cache.tile_size_bytes[1]/1e6:.3f} MB")
    print(f"  Using LRU: {cache.policy is not None}")
    print()
    
    # Create sample cache actions (prefetch decisions)
    cache_actions = [
        {
            'video': 0,
            'gop': 0,
            'tiles': np.array([1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
        },
        {
            'video': 1,
            'gop': 2,
            'tiles': np.array([0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
        },
        {
            'video': 0,
            'gop': 1,
            'tiles': np.array([1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
        },
    ]
    
    print(f"Processing {len(cache_actions)} cache actions...")
    print(f"Video requests: {[act['video'] for act in cache_actions]}\n")
    
    # Process cache prefetching
    cache_matrix = cache.process_cache_prefetching(cache_actions)
    
    print("Cache Results:")
    current_size = cache.get_current_capacity()
    print(f"  Final capacity used: {current_size/1e6:.2f} MB / {cache.max_capacity/1e6:.1f} MB")
    print(f"  Utilization: {(current_size/cache.max_capacity)*100:.1f}%")
    
    if cache.policy:
        stats = cache.policy.get_stats()
        print(f"  LRU stats: {stats['num_items']} items, {stats['utilization']:.1%} full")
    print()
    
    # Show cache content per video
    print("Cached tiles by video:")
    for vid in range(n_videos):
        base_cached = np.sum(cache_matrix[vid, 0, :, :])
        enh_cached = np.sum(cache_matrix[vid, 1, :, :])
        if base_cached > 0 or enh_cached > 0:
            print(f"  Video {vid}: Base={base_cached}/{n_tiles*n_tiles}, Enh={enh_cached}/{n_tiles*n_tiles}")
            if base_cached > 0:
                print(f"    Base layer tiles: {np.where(cache_matrix[vid, 0, :] == 1)[0].tolist()}")
            if enh_cached > 0:
                print(f"    Enh layer tiles:  {np.where(cache_matrix[vid, 1, :] == 1)[0].tolist()}")
    print()
    
    print("Testing tile access (updates LRU):")
    print(f"  Accessing tile (0, 0, 5): {cache.get_tile(0, 0, 5, 0)}")
    print(f"  Accessing tile (1, 0, 3): {cache.get_tile(1, 0, 3, 2)}")
    print(f"  Accessing non-cached (5, 0, 0): {cache.get_tile(5, 0, 0, 0)}")

===== Cache Storage Test with LRU Policy (Computed Matrix) =====
Configuration:
  Grid size: 4x4 = 16 tiles
  Layers: 2
  Videos: 10
  Capacity: 10.0 MB
  Base layer tile size: 0.125 MB
  Enhancement tile size: 0.938 MB
  Using LRU: True

Processing 3 cache actions...
Video requests: [0, 1, 0]

Cache Actions:
[{'video': 0, 'gop': 0, 'tiles': array([1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])}, {'video': 1, 'gop': 2, 'tiles': array([0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])}, {'video': 0, 'gop': 1, 'tiles': array([1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])}]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Cache Results:
  Final capacity used: 9.94 MB / 10.0 MB
  Utilization: 99.4%
  LRU stats: 47 items, 99.4% full

Cached tiles by video:
  Video 0: Base=26/16, Enh=2/16
    Base layer tiles: [0, 1, 2, 3, 4, 5, 6, 6, 7, 7, 8, 8, 9, 9, 10, 10, 11, 11, 12, 12, 13, 13, 14, 14, 15, 15]
    Enh layer tiles:  [0, 2]
  Video 1: Base=16/16, Enh=3/16
    Base layer tiles: [0, 1, 2, 3, 4, 5, 6, 7